# Synthetic valley: building the PEST interface

This notebook builds a PEST++ interface around the coastal synthetic-valley base model with [pyEMU](https://github.com/pypest/pyemu) and [pypestutils](https://github.com/pypest/pypestutils) (PPU). Hydraulic conductivity, vertical anisotropy, specific storage, and recharge are parameterized with pilot points and a geostatistical prior, recharge also with one multiplier per stress period, and the stream and coastal conductances with constants. The observations are the true heads at the observation wells, with measurement noise that is correlated in time. The companion notebook [`synthetic-valley-f-pestpp-ies`](synthetic-valley-f-pestpp-ies.ipynb) runs the iterative ensemble smoother PEST++ IES on this interface and inspects the results.
The workflow follows concepts and progression of the [GMDSI tutorial notebooks](https://github.com/gmdsi/GMDSI_notebooks).

## What this notebook covers

By the end of this notebook you will be able to:

- set up a PEST interface for a MODFLOW 6 model with `pyemu.utils.PstFrom`,
- design a geostatistical structure for pilot points and see what it implies for the prior,
- add pilot-point, constant, and time-varying multiplier parameters,
- fill the observations from the truth and design time-correlated measurement noise, and
- draw the prior parameter ensemble and the noise ensemble, and test the interface with PEST++ IES.

## Imports

`ipywidgets` provides the sliders used to design the prior and the noise. `PestUtilsLib` is PPU's library of PEST utilities, used here to place and krige the pilot points; it logs every call it makes, and a `NullHandler` keeps that out of the notebook. `sv_notebook_helpers` holds the location of the model files and `true_observations()`, which reads every observed series in the truth. The pilot points are also written to shapefiles, for GIS, and `pyshp` warns that the long parameter names are truncated in them; the interface does not read them, so that warning and a pandas deprecation warning raised inside pyEMU are silenced.

In [ ]:
%matplotlib inline
import logging
import pathlib as pl
import warnings

import flopy
import ipywidgets as widgets
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyemu
import pypestutils.helpers
from pypestutils.pestutilslib import PestUtilsLib
from sv_notebook_helpers import (
    BASE_MODEL,
    make_homogeneous,
    stream_flux_predictions,
    true_observations,
)

warnings.filterwarnings("ignore", module="shapefile")
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("PestUtilsLib").addHandler(logging.NullHandler())

## Write the model with external files

PEST++ adjusts the model inputs by rewriting them, so each array and list needs to be in its own file. `set_all_data_external()` writes every array and every stress period list to a text file named after the package and the array or period, for example `sv.npf_k_layer1.txt`. Run the model once so that the output files exist when the observations are added. The base model reads recharge as a list of cells, and pilot points need an array, so the RCH package is replaced with its array form holding the same rates; the heads match the list form to within 0.0001 m. Then `make_homogeneous()`, from the notebook helpers, gives each layer one value of K, vertical anisotropy, and storage, the geometric mean of its active cells, and each stress period one recharge rate, leaving the lake as it is. The parameters, not the base model, then make any pattern in space.

In [ ]:
name = "sv"
org_ws = pl.Path("models/sv-ies-org")  # the model with external files
template_ws = pl.Path("models/sv-ies-template")  # the PEST interface

sim = flopy.mf6.MFSimulation.load(
    sim_name=name, sim_ws=BASE_MODEL, write_headers=False, verbosity_level=0
)
sim.set_sim_path(org_ws)
gwf = sim.get_model(name)

# replace the list recharge with array recharge, which pilot points need
nrow, ncol = gwf.dis.nrow.data, gwf.dis.ncol.data
recharge = {}
for kper, rec in gwf.rch.stress_period_data.get_data().items():
    rate = np.zeros((nrow, ncol))
    for cellid, value in zip(rec["cellid"], rec["recharge"]):
        rate[cellid[1], cellid[2]] = value  # every recharge cell is in layer 1
    recharge[kper] = rate
gwf.remove_package("rch")
flopy.mf6.ModflowGwfrcha(gwf, recharge=recharge, pname="rch", filename="sv.rch")
make_homogeneous(gwf)  # one value per layer of each property, one recharge rate

# a looser closure than the model's, 0.0001 m, saves iterations and changes
# heads by far less than the noise
ims = sim.get_package("ims")
ims.complexity = "moderate"
ims.outer_dvclose = 1.0e-4
ims.outer_maximum = 500
ims.inner_dvclose = 1.0e-5

sim.set_all_data_external(check_data=False)
sim.write_simulation(silent=True)
success, buff = sim.run_simulation(silent=True)
assert success, "MODFLOW 6 did not terminate normally"
sorted(p.name for p in org_ws.glob("sv.npf*"))

## Start the PEST interface

`PstFrom` copies the model to `template_ws` and records every parameter and observation added to it. The model grid is its spatial reference, which locates the pilot points. MODFLOW 6 counts layers, rows, and columns from 1, so `zero_based=False`. `PstFrom` logs every step to `PstFrom.log` in `template_ws`; `echo=False` keeps the log out of the notebook.

In [ ]:
pf = pyemu.utils.PstFrom(
    original_d=org_ws,
    new_d=template_ws,
    remove_existing=True,
    longnames=True,
    spatial_reference=gwf.modelgrid,
    zero_based=False,
    start_datetime="1962-01-01",
    echo=False,
)

## Observations

The model writes five observation files: the water-table and lower-aquifer heads at the 17 observation wells, the flows between the aquifer and the stream and the sea, and the heads at the pumping and scenario wells. `add_observations()` reads each file, writes an instruction file, and names each value after its file, column, and time, so the names carry the information needed to set its value and weight later. The observation group is the file prefix, except for the steady-state heads, which get a group of their own below. The **water budget** that MODFLOW 6 writes to `sv-budget.csv` is added too, in group `bud`: every flow in and out of the model by package (storage, recharge, evapotranspiration, the stream, the coast, the wells) at every output time, with the totals and the percent discrepancy. The truth has no budget, so these observations get no weight; they come back in the ensemble files, so the prior and posterior budgets can be compared.

The **predictions** are three numbers the model run does not write directly: the groundwater discharge to the stream in the steady state, and the smallest groundwater discharge to the stream in any year of the history, and in any year of the future. `stream_flux_predictions()`, from the notebook helpers, reads the stream flows MODFLOW 6 wrote and writes the three to `sv.predictions.csv`. `add_py_function()` copies it into the forward run script, to run after MODFLOW 6 (`is_pre_cmd=False`), and running it once now makes the file that `add_observations()` reads.

In [ ]:
obs_files = {
    "sv.gwf.wt.csv": "wt",
    "sv.gwf.aq.csv": "aq",
    "sv.riv.obs.csv": "riv",
    "sv.ghb.obs.csv": "coast",
    "sv.gwf.scenario.csv": "scen",
}
for f, prefix in obs_files.items():
    df = pd.read_csv(org_ws / f, index_col=0)
    pf.add_observations(
        f,
        insfile=f + ".ins",
        index_cols="time",
        use_cols=list(df.columns),
        prefix=prefix,
        obsgp=[prefix] * len(df.columns),
    )

# the water budget at every output time, every flow in and out of the model
# by package, from the budget csv that MODFLOW 6 writes (BUDGETCSV in OC);
# no weight, so IES does not fit them
budget = pd.read_csv(org_ws / "sv-budget.csv", index_col=0)
pf.add_observations(
    "sv-budget.csv",
    insfile="sv-budget.csv.ins",
    index_cols="time",
    use_cols=list(budget.columns),
    prefix="bud",
    obsgp=["bud"] * len(budget.columns),  # one group
)

# the predictions: the groundwater discharge to the stream (m3/d) in the steady
# state, and the smallest in the history and in the future, computed from the stream flows after each run
stream_flux_predictions(
    template_ws / "sv.riv.obs.csv", out_file=template_ws / "sv.predictions.csv"
)
pf.add_py_function(
    "sv_notebook_helpers.py",
    "stream_flux_predictions('sv.riv.obs.csv')",
    is_pre_cmd=False,
)
pf.add_observations(
    "sv.predictions.csv",
    index_cols="prediction",
    use_cols=["discharge"],
    prefix="pred",
    obsgp="pred",
)

## Design the geostatistical prior

Pilot points are a small set of points where the hydraulic conductivity multiplier is adjusted; kriging spreads their values to every cell. The kriging and the prior both come from a **geostatistical structure**: a variogram with a range `a` (the distance over which values are correlated), an anisotropy ratio and bearing (whether the correlation is longer in one direction), and a variogram type.

The sliders do what the interface will do. PPU places the pilot points every `pp_space` rows and columns of the active cells (`get_2d_pp_info_structured_grid()`, reading the grid from the binary grid file MODFLOW 6 wrote); four realizations of the log10 multiplier are drawn at the pilot points from the structure's covariance matrix (`covariance_matrix()`); and PPU kriges each one to the cells with the same two calls the interface makes at run time, `calc_kriging_factors_2d()` and `krige_using_file()`. The structure has a sill (`contribution`) of 1, as pyEMU expects; the multiplier bounds set the spread when the prior is drawn, with the bounds taken as 6 standard deviations, so bounds of 0.1 to 10 give a standard deviation of 1/3 in log10 units, which scales the values drawn. Each realization is shown as the layer 1 K it gives, the homogeneous K of the layer times the kriged multiplier, on a log scale one factor of 10 either side of that K. The random numbers behind the four realizations stay the same, so each slider shows what the structure does to the same draws.

In [ ]:
grid = gwf.modelgrid
idomain = gwf.dis.idomain.array
lake = gwf.npf.k.array[0] > 1000.0
zone = (idomain[0] > 0).astype(int)  # the layer 1 cells the pilot points cover
zone[lake] = 0
org_k = gwf.npf.k.array[0][zone > 0].mean()  # the homogeneous K of layer 1
grb_file = str(org_ws / "sv.dis.grb")  # the binary grid file, for PPU
width = 17.15 / 2.54  # figure width, inches

VARIOGRAMS = {  # pyEMU variogram class, PPU variogram name
    "exponential": (pyemu.geostats.ExpVario, "exp"),
    "spherical": (pyemu.geostats.SphVario, "spher"),
    "gaussian": (pyemu.geostats.GauVario, "gauss"),
}
pp_bounds = (0.1, 10.0)  # multiplier bounds of the pilot points
sd_log = (np.log10(pp_bounds[1]) - np.log10(pp_bounds[0])) / 6.0
lib = PestUtilsLib()


def make_geostruct(a, anisotropy=1.0, bearing=0.0, variogram="exponential"):
    v = VARIOGRAMS[variogram][0](
        contribution=1.0, a=a, anisotropy=anisotropy, bearing=bearing
    )
    return pyemu.geostats.GeoStruct(variograms=v, transform="log")


def krige_to_grid(pp, draws, a, anisotropy, bearing, variogram):
    # the kriging factors from the pilot points to the active cells, once, then
    # one interpolation per draw; cells outside the zone get NaN
    factors = str(org_ws / "design.fac")
    lib.calc_kriging_factors_2d(
        pp.x.values,
        pp.y.values,
        1,  # source points and zone
        grid.xcellcenters.ravel(),
        grid.ycellcenters.ravel(),
        zone.ravel(),
        VARIOGRAMS[variogram][1],
        "ordinary",
        a,
        anisotropy,
        bearing,
        1.0e10,
        50,
        1,  # search radius, max and min points
        factors,
        "text",
    )
    fields = []
    for values in draws:
        result = lib.krige_using_file(
            factors, "text", zone.size, "ordinary", "none", values, 0.0, np.nan
        )
        fields.append(np.ma.masked_invalid(result["targval"].reshape(zone.shape)))
    lib.free_all_memory()
    return fields


def show_prior(
    pp_space=4, a=1000.0, anisotropy=1.0, bearing=0.0, variogram="exponential"
):
    pp = pypestutils.helpers.get_2d_pp_info_structured_grid(
        pp_space, grb_file, array_dict={"zone": zone}
    )
    gs = make_geostruct(a, anisotropy, bearing, variogram)
    cov = gs.covariance_matrix(pp.x.values, pp.y.values, names=pp.ppname.tolist())
    rng = np.random.default_rng(1)
    draws = sd_log * rng.multivariate_normal(np.zeros(len(pp)), cov.x, size=4)
    fields = krige_to_grid(pp, draws, a, anisotropy, bearing, variogram)
    # the K of each realization: the homogeneous K times the kriged multiplier
    k_fields = [org_k * 10.0**field for field in fields]
    k_points = org_k * 10.0**draws
    norm = mpl.colors.LogNorm(org_k / 10.0, org_k * 10.0)

    fig, axes = plt.subplots(
        1, 4, figsize=(width, 0.5 * width), layout="constrained", sharey=True
    )
    for n, ax in enumerate(axes):
        mm = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=0)
        pa = mm.plot_array(k_fields[n], cmap="viridis", norm=norm)
        ax.scatter(
            pp.x,
            pp.y,
            c=k_points[n],
            cmap="viridis",
            norm=norm,
            s=8,
            ec="k",
            lw=0.3,
        )
        ax.set_aspect("equal")
        ax.set_title(f"Realization {n + 1}")
    fig.suptitle(f"{len(pp)} pilot points")
    fig.colorbar(pa, ax=axes, shrink=0.8, label="Layer 1 K (m/d)")
    plt.show()


widgets.interact(
    show_prior,
    pp_space=(2, 10, 1),
    a=(200.0, 4000.0, 100.0),
    anisotropy=(1.0, 5.0, 0.5),
    bearing=(0.0, 180.0, 15.0),
    variogram=list(VARIOGRAMS),
);

Set the spacing and the structure to use. The valley drains from north to south, so a mild anisotropy along the valley is plausible.

In [ ]:
pp_space = 4
grid_gs = make_geostruct(a=1000.0, anisotropy=1.0, bearing=0.0, variogram="exponential")
grid_gs

## Hydraulic conductivity, anisotropy, and storage parameters

Each layer gets, as multipliers on the array in the file:

- **pilot points** on the horizontal K, the vertical anisotropy `k33` (the model's NPF package uses `K33OVERK`, so this array is the ratio of vertical to horizontal K), and the specific storage `ss`, every `pp_space` cells, with the structure designed above (`geostruct`), and
- a **constant** on each of the three, so the whole layer can shift together.

The zone array limits the parameters to the active cells of the layer, and to the cells outside the lake in layer 1, so the lake's very high K is left alone. With `try_use_ppu`, PPU computes the kriging factors now and interpolates the pilot points to the grid in every model run. All parameters are log transformed (the default for multipliers), with their bounds used as the prior range.

In [ ]:
nlay = gwf.dis.nlay.data
for k in range(nlay):
    layer_zone = (idomain[k] > 0).astype(int)
    if k == 0:
        layer_zone[lake] = 0
    layer = f"layer{k + 1}"

    # pilot points on K, k33, and ss
    for array, base, bounds in [
        ("npf_k", f"k{k + 1}pp", pp_bounds),
        ("npf_k33", f"k33{k + 1}pp", (0.2, 5.0)),
        ("sto_ss", f"ss{k + 1}pp", (0.2, 5.0)),
    ]:
        pf.add_parameters(
            f"sv.{array}_{layer}.txt",
            par_type="pilotpoints",
            par_name_base=base,
            pargp=base,
            zone_array=layer_zone,
            lower_bound=bounds[0],
            upper_bound=bounds[1],
            geostruct=grid_gs,
            pp_options={"pp_space": pp_space, "try_use_ppu": True},
        )

    # a constant on each, so the whole layer can shift together
    for array, base, bounds in [
        ("npf_k", f"k{k + 1}cn", (0.2, 5.0)),
        ("npf_k33", f"k33{k + 1}cn", (0.1, 10.0)),
        ("sto_ss", f"ss{k + 1}cn", (0.1, 10.0)),
    ]:
        pf.add_parameters(
            f"sv.{array}_{layer}.txt",
            par_type="constant",
            par_name_base=base,
            pargp=base,
            zone_array=layer_zone,
            lower_bound=bounds[0],
            upper_bound=bounds[1],
        )

## Recharge and conductance parameters

Recharge is now an array for each stress period, in files like `sv.rch_recharge_1.txt`. Each period gets its own constant multiplier. The periods share the parameter group `rch`, and each parameter carries the date of the end of its period, so that a temporal geostatistical structure, `temporal_gs`, can correlate the multipliers of nearby years when the prior is drawn: an exponential variogram whose range `a` is in days. One set of **pilot points**, the group `rchpp`, multiplies the recharge of every period, so the constants carry the changes from year to year and the pilot points carry the pattern in space. The lake cells are left out of the pilot points.

The conductances of the stream (RIV) and the coast (GHB) each get one constant multiplier.

In [ ]:
perlen = sim.tdis.perioddata.array["perlen"]
dates = pd.Timestamp("1962-01-01") + pd.to_timedelta(np.cumsum(perlen), unit="D")
temporal_gs = pyemu.geostats.GeoStruct(
    variograms=pyemu.geostats.ExpVario(contribution=1.0, a=36500.0), transform="log"
)
for kper, date in enumerate(dates):
    pf.add_parameters(
        f"sv.rch_recharge_{kper + 1}.txt",
        par_type="constant",
        par_name_base="rch",
        pargp="rch",
        lower_bound=0.5,
        upper_bound=2.0,
        datetime=date,
        geostruct=temporal_gs,
    )

# one set of pilot points multiplies the recharge of every period
pf.add_parameters(
    [f"sv.rch_recharge_{kper + 1}.txt" for kper in range(len(dates))],
    par_type="pilotpoints",
    par_name_base="rchpp",
    pargp="rchpp",
    zone_array=zone,
    lower_bound=0.5,
    upper_bound=2.0,
    geostruct=grid_gs,
    pp_options={"pp_space": pp_space, "try_use_ppu": True},
)

for package, base in [("riv", "rivcond"), ("ghb", "coastcond")]:
    pf.add_parameters(
        f"sv.{package}_stress_period_data_1.txt",
        par_type="constant",
        par_name_base=base,
        pargp=base,
        index_cols=[0, 1, 2],
        use_cols=[4],
        lower_bound=0.1,
        upper_bound=10.0,
    )

## Property observations

The K, k33, and ss arrays that the forward run writes are added as observations too, one for each active cell of each layer, so that every realization's properties come back in the ensemble files and can be mapped without running the model again. They get no weight, so IES does not fit them. The lake cells of layer 1 are left out, as they are from the parameters. Specific yield is left out: every cell of the model is confined (`ICONVERT` is 0), so MODFLOW 6 does not use it, and it has no parameters.

In [ ]:
# the property arrays of every layer, as observations with no weight
for k in range(nlay):
    layer_zone = (idomain[k] > 0).astype(int)
    if k == 0:
        layer_zone[lake] = 0  # the lake's very high K is not a property to map
    layer = f"layer{k + 1}"
    for array, prefix in [
        ("npf_k", f"k{k + 1}"),
        ("npf_k33", f"k33{k + 1}"),
        ("sto_ss", f"ss{k + 1}"),
    ]:
        pf.add_observations(
            f"sv.{array}_{layer}.txt",
            prefix=prefix,
            obsgp=prefix,
            zone_array=layer_zone,
        )

## Build the control file

The forward run is the script `forward_run.py` that `PstFrom` writes: it applies the multipliers to the model input files, then runs the command in `mod_sys_cmds`, here `mf6`. `build_pst()` writes the template and instruction files and the control file `pest.pst`, and returns it as a `pyemu.Pst`. The parameter names carry the group, and the row `i` and column `j` of the pilot points, counted from 0, which the map below uses to locate them on the grid.

In [ ]:
pf.mod_sys_cmds.append("mf6")
pst = pf.build_pst("pest.pst")
par = pst.parameter_data
print(f"{pst.npar} parameters in {len(pst.par_groups)} groups, {pst.nobs} observations")
par.loc[par.ptype == "pp", "pargp"].value_counts().sort_index()

## Observed values and weights

The truth comes from the Voronoi-grid advanced model, which `true_observations()` returns as one column per observation, named like the model's observation columns, by time. Each observation's `usecol` and `time`, parsed from its name, look up its true value.

The first 11 output times, through the end of stress period 11 (day 19,723), are the **history** that IES matches: the steady state and the first 10 years of pumping. The steady-state heads, at the first output time, are put in a group of their own, `ssheads`, so that PEST++ reports their part of phi apart from the transient heads. Stress period 12 starts a new well, and the remaining 10 years are the **forecast**, with a weight of 0 so that IES sees but does not fit them. The flows and scenario heads are forecasts too. The true predictions come from the truth's flow between the aquifer and the stream, and the option `forecasts` tells PEST++ which observations they are, so that it reports their uncertainty. The history heads have a measurement standard deviation of 0.1 m, which `standard_deviation` records and the weight (its inverse) uses.

In [ ]:
truth = true_observations()
obs = pst.observation_data
obs["time"] = obs.time.astype(float)
series = obs.oname.isin(list(obs_files.values()))  # not the predictions
obs.loc[series, "obsval"] = [
    truth.loc[t, u] for u, t in zip(obs.usecol[series], obs.time[series])
]
# the steady-state heads are a group of their own, so that phi reports them apart
steady = obs.oname.isin(["wt", "aq"]) & (obs.time == 16071.0)
obs.loc[steady, "obgnme"] = "ssheads"

history = obs.oname.isin(["wt", "aq"]) & (obs.time <= 19723.0)
noise_sd = 0.1
obs["weight"] = 0.0
obs.loc[history, "weight"] = 1.0 / noise_sd
obs.loc[obs.obgnme == "bud", "weight"] = 0.0  # the budget is not observed
obs["standard_deviation"] = noise_sd

# the true predictions: the discharge to the stream in the steady state, the
# first output time, and the smallest discharge in the history and in the future
true_discharge = -truth["riv-swgw"]  # the truth's flow is negative to the stream
predictions = obs.loc[obs.oname == "pred"]
for name, prediction in zip(predictions.obsnme, predictions.prediction):
    if prediction == "steadystate":
        obs.loc[name, "obsval"] = true_discharge.iloc[0]
    elif prediction == "minhistory":
        obs.loc[name, "obsval"] = true_discharge.loc[:19723.0].min()
    else:
        obs.loc[name, "obsval"] = true_discharge.loc[19723.0:].iloc[1:].min()
pst.pestpp_options["forecasts"] = ",".join(predictions.obsnme)
print(f"{pst.nnz_obs} history observations of {pst.nobs}")
print(obs.loc[predictions.obsnme, ["prediction", "obsval"]])

## Design the noise autocorrelation

Measurement noise is usually drawn independently for each observation. Here the errors in an annual head series are **correlated in time**: a well that reads high this year tends to read high next year, from a survey or screen error that persists, and from the parts of the system the model simplifies. Correlated noise realizations keep IES from treating every year as an independent piece of information.

The noise at one well is a Gaussian series with a standard deviation `sd` and an exponential correlation in time with a range `a` in days. The range slider is in days, and its smallest value is the shortest stress period, because the noise is drawn at the output times, one per stress period, and a range shorter than the spacing of those times would make every year's noise independent. The sliders show the true heads at four water table wells, in red, with five noise realizations added to each, in gray, at the 11 history times, which are one year apart.

In [ ]:
times = np.sort(obs.loc[history, "time"].unique())
years = (times - times[0]) / 365.25
wells = ["wt01-i2-j17", "wt05-i14-j11", "wt09-i19-j11", "wt14-i30-j6"]
min_perlen = perlen.min()  # the shortest stress period length (days)


def show_noise(sd=0.1, a=730.0):
    gs = pyemu.geostats.GeoStruct(
        variograms=pyemu.geostats.ExpVario(contribution=sd**2, a=a)
    )
    cov = gs.covariance_matrix(
        times, np.zeros_like(times), names=[str(t) for t in times]
    )
    rng = np.random.default_rng(2)

    fig, axes = plt.subplots(
        2, 2, figsize=(width, 0.7 * width), layout="constrained", sharex=True
    )
    for ax, well in zip(axes.flat, wells):
        # the true heads of this well, in time order
        true = obs.loc[history & (obs.usecol == well)].sort_values("time").obsval.values
        # five noise series, independent of the other wells
        noise = rng.multivariate_normal(np.zeros(len(times)), cov.x, size=5)
        for series in noise:
            ax.plot(years, true + series, color="0.6", lw=0.8)
        ax.plot(years, true, color="red", lw=1.5)
        ax.set_ylim(true.mean() - 1.5, true.mean() + 1.5)
        ax.set_title(well.split("-")[0].upper())
    axes[0, 0].set_ylabel("Head (m)")
    axes[1, 0].set_ylabel("Head (m)")
    axes[1, 0].set_xlabel("Years since the steady state")
    axes[1, 1].set_xlabel("Years since the steady state")
    plt.show()


a_slider = widgets.FloatSlider(
    value=2.0 * min_perlen,
    min=min_perlen,
    max=30.0 * min_perlen,
    step=min_perlen / 10.0,
    description="a (days)",
)
widgets.interact(show_noise, sd=(0.02, 0.5, 0.02), a=a_slider);

Set the range to use. The standard deviation is already in the observation data.

In [ ]:
noise_a = 3650.0

## Draw the noise ensemble

`pyemu.helpers.autocorrelated_draw()` takes a dictionary of geostatistical structures and the observations each one correlates, and scales the correlation by each observation's standard deviation. Each well's series is a separate structure, so that noise is correlated along a well's record and independent between wells. The result is an ensemble of noisy observation values, one realization per row, saved for IES. The plot shows the realizations at one well with the true values.

In [ ]:
num_reals = 500  # just to have a lot...
struct_dict = {
    pyemu.geostats.GeoStruct(
        variograms=pyemu.geostats.ExpVario(contribution=1.0, a=noise_a), name=well
    ): df.obsnme.tolist()
    for well, df in obs.loc[history].groupby("usecol")
}
oe = pyemu.helpers.autocorrelated_draw(
    pst, struct_dict, time_distance_col="time", num_reals=num_reals, verbose=False
)
oe.to_csv(template_ws / "obs_noise.csv")

well = obs.loc[history & (obs.usecol == "wt05-i14-j11")].sort_values("time")
with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(figsize=(width, 0.4 * width), layout="constrained")
    years = (well.time - well.time.min()) / 365.25
    ax.plot(years, oe.loc[:, well.obsnme].values.T, color="0.7", lw=0.5)
    ax.plot(years, well.obsval, "o-", color="red", ms=4, lw=1.0, label="True head")
    flopy.plot.styles.xlabel(ax=ax, label="Years since the steady state")
    flopy.plot.styles.ylabel(ax=ax, label="Head (m)")
    flopy.plot.styles.heading(
        ax=ax, heading=f"{well.usecol.iloc[0]}: noise realizations"
    )
    flopy.plot.styles.graph_legend(ax=ax, loc="upper right")

## Draw the prior parameter ensemble

`pf.draw()` draws the parameter ensemble from the bounds and the geostatistical structures: the pilot points of each group are correlated in space by `grid_gs` (`use_specsim` draws them with spectral simulation), the recharge multipliers are correlated in time by `temporal_gs`, and the other constants are independent. `enforce()` clips any draw that falls outside its bounds. The plots show the layer 1 pilot points of one realization, and the recharge multipliers of every realization with five of them picked out.

In [ ]:
pe = pf.draw(num_reals=num_reals, use_specsim=True)
pe.enforce()
pe.to_csv(template_ws / "prior_pe.csv")

pp = par.loc[par.pargp == "k1pp"]
i, j = pp.i.astype(int), pp.j.astype(int)
rch = par.loc[par.pargp == "rch"]
kper = rch.inst.astype(int) + 1  # the instance counts the add_parameters() calls
with flopy.plot.styles.USGSMap():
    fig, (ax1, ax2) = plt.subplots(
        1, 2, figsize=(width, 0.55 * width), layout="constrained", width_ratios=[1, 1.4]
    )
    mm = flopy.plot.PlotMapView(model=gwf, ax=ax1, layer=0)
    mm.plot_grid(lw=0.2, color="0.8")
    sc = ax1.scatter(
        grid.xcellcenters[i, j],
        grid.ycellcenters[i, j],
        c=np.log10(pe.loc[pe.index[0], pp.parnme].astype(float)),
        cmap="RdBu_r",
        vmin=-1.0,
        vmax=1.0,
        s=25,
    )
    ax1.set_aspect("equal")
    ax1.set_xlabel("x (m)")
    ax1.set_ylabel("y (m)")
    fig.colorbar(
        sc, ax=ax1, orientation="horizontal", shrink=0.9, label="log10 multiplier"
    )
    flopy.plot.styles.heading(ax=ax1, letter="A", heading="Layer 1 pilot points")

    ax2.plot(kper, pe.loc[:, rch.parnme].values.T, color="0.85", lw=0.5)
    ax2.plot(kper, pe.loc[pe.index[:5], rch.parnme].values.T, marker="o", ms=3, lw=1.0)
    ax2.axhline(1.0, color="red", lw=1.0)
    ax2.set_yscale("log")
    ax2.set_xlabel("Stress period")
    ax2.set_ylabel("Recharge multiplier")
    flopy.plot.styles.heading(ax=ax2, letter="B", heading="Recharge, all realizations")

## PEST++ IES settings and a test run

The `pestpp_options` tell PEST++ IES to start from the ensembles just drawn, to use `num_reals` realizations, to test its step lengths on a subset of 10 of them, to drop realizations whose fit is more than 1.7 standard deviations worse than the mean, and never to give up on a model run that is slower than the others (`overdue_giveup_fac`): on a busy computer a slow run is not a failed one. `noptmax=0` makes a single run with the initial parameter values (the base model), which checks that the forward run and the instruction files work. `pst.phi` is the weighted sum of squared residuals of that run.

In [ ]:
pst.pestpp_options["ies_parameter_ensemble"] = "prior_pe.csv"
pst.pestpp_options["ies_observation_ensemble"] = "obs_noise.csv"
pst.pestpp_options["ies_num_reals"] = num_reals
pst.pestpp_options["ies_subset_size"] = 10
pst.pestpp_options["ies_bad_phi_sigma"] = 1.7
pst.pestpp_options["overdue_giveup_fac"] = 1.0e30  # never kill slow runs
pst.control_data.noptmax = 0
pst.write(template_ws / "pest.pst", version=2)

pyemu.os_utils.run("pestpp-ies pest.pst", cwd=template_ws)
pst = pyemu.Pst(str(template_ws / "pest.pst"))
print(f"phi of the base model {pst.phi:.0f} for {pst.nnz_obs} observations")
pst.res.loc[pst.nnz_obs_names, ["group", "measured", "modelled", "residual"]].describe()

If every residual were as large as the noise, phi would equal the number of observations; the phi of the homogeneous model shows how far its heads are from the truth.

Set the number of iterations for the companion notebook and write the control file.

In [ ]:
pst.control_data.noptmax = 3
pst.write(template_ws / "pest.pst", version=2)

## Run one realization

To see what a realization of the prior does to the model inputs, write its parameter values into a copy of the control file, `test.pst`, and run it once with `noptmax=0`. The forward run applies its multipliers, and the property observations bring the K, k33, and ss of every layer back in `test.base.obs.csv`. The copy is read from `pest.pst` so that `pest.pst` keeps the initial parameter values: the `base` realization of IES starts from them.

In [ ]:
real = pe.index[1]  # the realization to run
test_pst = pyemu.Pst(str(template_ws / "pest.pst"))
test_pst.parameter_data.loc[pe.columns, "parval1"] = pe.loc[real, pe.columns].values
test_pst.control_data.noptmax = 0
test_pst.write(template_ws / "test.pst", version=2)

pyemu.os_utils.run("pestpp-ies test.pst", cwd=template_ws)
real_values = pd.read_csv(template_ws / "test.base.obs.csv", index_col=0).iloc[0]
print(f"realization {real}: {len(real_values)} simulated values")

In [ ]:
obs = pst.observation_data
nrow, ncol = gwf.dis.nrow.data, gwf.dis.ncol.data


def property_array(prefix):
    # the property observations of one array, back on the grid; i and j count
    # from 0, and cells outside the zone stay empty
    rows = obs.loc[obs.oname == prefix]
    arr = np.full((nrow, ncol), np.nan)
    arr[rows.i.astype(int), rows.j.astype(int)] = real_values.loc[rows.obsnme].values
    return np.ma.masked_invalid(arr)


k_arrays = [property_array(f"k{k + 1}") for k in range(nlay)]
k33_arrays = [property_array(f"k33{k + 1}") for k in range(nlay)]
ss_arrays = [property_array(f"ss{k + 1}") for k in range(nlay)]

# one color scale for each property, shared by the layers
k_norm = mpl.colors.LogNorm(
    min(a.min() for a in k_arrays), max(a.max() for a in k_arrays)
)
k33_norm = mpl.colors.LogNorm(
    min(a.min() for a in k33_arrays), max(a.max() for a in k33_arrays)
)
ss_norm = mpl.colors.LogNorm(
    min(a.min() for a in ss_arrays), max(a.max() for a in ss_arrays)
)

with flopy.plot.styles.USGSMap():
    fig, axes = plt.subplots(
        3, nlay, figsize=(width, 1.05 * width), layout="constrained", sharey=True
    )
    for k in range(nlay):
        mm = flopy.plot.PlotMapView(model=gwf, ax=axes[0, k], layer=k)
        pk = mm.plot_array(k_arrays[k], cmap="viridis", norm=k_norm)
        mm = flopy.plot.PlotMapView(model=gwf, ax=axes[1, k], layer=k)
        pk33 = mm.plot_array(k33_arrays[k], cmap="viridis", norm=k33_norm)
        mm = flopy.plot.PlotMapView(model=gwf, ax=axes[2, k], layer=k)
        pss = mm.plot_array(ss_arrays[k], cmap="viridis", norm=ss_norm)
        axes[0, k].set_title(f"Layer {k + 1}")
    for ax in axes.flat:
        ax.set_aspect("equal")
        ax.set_xticks([])
        ax.set_yticks([])
    fig.colorbar(pk, ax=axes[0, :], shrink=0.8, label="K (m/d)")
    fig.colorbar(pk33, ax=axes[1, :], shrink=0.8, label="k33 (vertical / horizontal K)")
    fig.colorbar(pss, ax=axes[2, :], shrink=0.8, label="Ss (1/m)")
    fig.suptitle(f"Prior realization {real}")

## Recap

You wrote the base model with external files, built a PEST interface on it with `PstFrom`, designed a geostatistical structure for the pilot points on hydraulic conductivity and saw the prior fields it implies, added constant and time-correlated multipliers for the other inputs, filled the observations from the truth, designed and drew time-correlated measurement noise, drew the prior parameter ensemble, tested the interface with one run of PEST++ IES, and mapped the properties of one prior realization from the property observations. The companion notebook, [`synthetic-valley-f-pestpp-ies`](synthetic-valley-f-pestpp-ies.ipynb), runs the iterations.